Loading the dataset


In [ ]:
import kagglehub
import os
import pandas as pd
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
file_path = os.path.join(path, "IMDB Dataset.csv")
df = pd.read_csv(file_path)


Imputing target values to binary(0/1)

In [ ]:
df['sentiment']=df['sentiment'].replace({'positive':1,'negative':0})

creating train test split

In [ ]:
from sklearn.model_selection import train_test_split
review_train,review_test,sentiment_train,sentiment_test=train_test_split(
    df['review'].tolist(),
    df['sentiment'].tolist(),
    random_state=2,
)

tokenizing reviews

In [ ]:
from transformers import AutoTokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_encodings = tokenizer(
    list(review_train),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt",
)
test_encodings = tokenizer(
    list(review_test),
    truncation=True,
    padding=True,
    max_length=256,
    return_tensors="pt",
)

Load into PyTorch dataset

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class Reviews(Dataset):
  def __init__(self,encodings,labels):
    self.encodings=encodings
    self.labels=labels
  def __len__(self):
    return len(self.labels)
  def __getitem__(self,idx):
    item = {key: val[idx] for key, val in self.encodings.items()}
    item["labels"] = torch.tensor(self.labels[idx])
    return item
train_data = Reviews(train_encodings,sentiment_train)
test_data = Reviews(test_encodings, sentiment_test)

train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
test_loader=DataLoader(test_data,batch_size=32)


Loading the classifier

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)
devtype=input("M for mac, W for windows: ").upper()
if devtype == "M":
  device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
elif devtype=="W":
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
else:
  device="cpu"
print(f"Device: {device}")
model.to(device)

Training loop


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)#small LR due to retraining
num_epochs = 3

model.train()
print(f"Beginning training with {num_epochs} epochs.")
for epoch in range(num_epochs):
    total_loss = 0
    print(f"Epoch {epoch+1}:")
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs} has avg training loss: {avg_loss:.4f}")


Evaluation loop

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

model.eval()
all_preds = []
all_labels = []
print("Beginning testing.")
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

print(f"Validation accuracy: {accuracy_score(all_labels, all_preds):.4f}")
print(classification_report(all_labels, all_preds, target_names=["negative", "positive"]))

Model evaluation metrics

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt
import torch.nn.functional as F

#Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["negative", "positive"], yticklabels=["negative", "positive"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

#ROC AUC curve
print(f"ROC-AUC: {roc_auc_score(all_labels, all_preds):.4f}")

#Texts misclassified
misclassified = [
    (text, true, pred)
    for text, true, pred in zip(review_test[:150], all_labels, all_preds)
    if true != pred
]
for text, true, pred in misclassified[:10]:
    print(f"TRUE={true} PRED={pred}: \n{text}")

Saving the model

In [ ]:
from huggingface_hub import login

login()
#Huggingface upload
model.push_to_hub("Rayhan-08/IMDBert")
tokenizer.push_to_hub("Rayhan-08/IMDBert")
#Local file save
model.save_pretrained("./models/IMDBert")
tokenizer.save_pretrained("./models/IMDBert")

To reload model for future use, run:
```python
model = AutoModelForSequenceClassification.from_pretrained("./models/IMDBert")

tokenizer = AutoTokenizer.from_pretrained("./models/IMDBert")
```
Or to load from HuggingFace:


```python
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("Rayhan-08/IMDBert")
tokenizer = AutoTokenizer.from_pretrained("Rayhan-08/IMDBert")
```

